In [ ]:
silver_df = spark.read.table('workspace.ecommerce.bronze_retail')

In [ ]:
#display first letter of the invoice to check the records which do not have C as prefix
from pyspark.sql.functions import *
silver_df_null = silver_df.filter(col("Quantity")<0 )
silver_df_null = silver_df_null.filter(~col("Invoice").startswith("C"))
display(silver_df_null)



In [ ]:
# Separating:
# 1. C-prefixed credit/return transactions
# 2. Normal positive-quantity transactions
# 3. Non-C-prefixed negative-quantity transactions

c_prefixed_sales_df = silver_df.filter(
    col("Invoice").startswith("C")
)

normal_sales_df = silver_df.filter(
    (~col("Invoice").startswith("C")) &
    (col("Quantity") > 0)
)

non_c_prefixed_df = silver_df.filter(
    (~col("Invoice").startswith("C")) &
    (col("Quantity") < 0)
)

# Individual counts
c_prefixed_count = c_prefixed_sales_df.count()
normal_sales_count = normal_sales_df.count()
non_c_prefixed_count = non_c_prefixed_df.count()
source_count = silver_df.count()

# Reconciliation
classified_total = (
    c_prefixed_count +
    normal_sales_count +
    non_c_prefixed_count
)

unclassified_count = source_count - classified_total

# Print results
print("=" * 60)
print("BASELINE TRANSACTION CLASSIFICATION")
print("=" * 60)

print(f"C-Prefixed Transactions      : {c_prefixed_count:,}")
print(f"Normal Positive Transactions : {normal_sales_count:,}")
print(f"Non-C Negative Transactions : {non_c_prefixed_count:,}")

print("-" * 60)
print(f"Total Classified Records     : {classified_total:,}")
print(f"Total Source Records         : {source_count:,}")
print(f"Unclassified / Difference    : {unclassified_count:,}")
print("=" * 60)

In [ ]:
#Profiling the normal_Sales_df
from pyspark.sql.functions import *
normal_sales_df.filter(col("Price")<0).display()
print(f"Total Negative Price Records: {normal_sales_df.filter(col('Price')<0).count()}")
normal_sales_df.filter(col("Customer_ID").isNull()).display()
print(f"Total Null Customer_id Records: {normal_sales_df.filter(col('Customer_ID').isNull()).count()}")
normal_sales_df.filter(col("Description").isNull()).display()
print(f"Total Null Desscription Records: {normal_sales_df.filter(col('Description').isNull()).count()}")

In [ ]:
# Identifying Bad Debt Adjustments
bad_debt_df = silver_df.filter(
    (~col("Invoice").startswith("C")) &
    (col("StockCode") == "B") &
    (col("Description") == "Adjust bad debt")
)

# Updating Normal Sales by excluding Bad Debt Adjustments
normal_sales_df = silver_df.filter(
    (~col("Invoice").startswith("C")) &
    (col("Quantity") > 0) &
    ~(
        (col("StockCode") == "B") &
        (col("Description") == "Adjust bad debt")
    )
)

# Keeping other classifications unchanged
c_prefixed_sales_df = silver_df.filter(
    col("Invoice").startswith("C")
)

non_c_prefixed_df = silver_df.filter(
    (~col("Invoice").startswith("C")) &
    (col("Quantity") < 0)
)

# Individual counts
c_prefixed_count = c_prefixed_sales_df.count()
normal_sales_count = normal_sales_df.count()
non_c_prefixed_count = non_c_prefixed_df.count()
bad_debt_count = bad_debt_df.count()
source_count = silver_df.count()

# Reconciliation
classified_total = (
    c_prefixed_count +
    normal_sales_count +
    non_c_prefixed_count +
    bad_debt_count
)

unclassified_count = source_count - classified_total

# Output
print("=" * 60)
print("REVISED TRANSACTION CLASSIFICATION")
print("=" * 60)

print(f"C-Prefixed Transactions      : {c_prefixed_count:,}")
print(f"Normal Sales                : {normal_sales_count:,}")
print(f"Non-C Negative Transactions : {non_c_prefixed_count:,}")
print(f"Bad-Debt Adjustments        : {bad_debt_count:,}")

print("-" * 60)
print(f"Total Classified Records    : {classified_total:,}")
print(f"Total Source Records        : {source_count:,}")
print(f"Unclassified / Difference   : {unclassified_count:,}")
print("=" * 60)

In [ ]:
#Working on the Duplicates records
d_count = normal_sales_df.distinct().count()
total_count = normal_sales_df.count()


dup_count = total_count - d_count
print(f"Distinct Count: {d_count:,}")
print(f"Total Count: {total_count:,}")
print(f"Total Duplicates: {dup_count:,}")


In [ ]:
#Finding the duplicate data

dup_df = normal_sales_df.groupBy("Invoice","StockCode","Description","Quantity","InvoiceDate","Price","Customer_ID","Country").count().filter(col("count")>1).orderBy(col("Customer_ID"))
display(dup_df)

#Removing the duplicate data
normal_sales_cleaned_df = normal_sales_df.dropDuplicates()
normal_sales_cleaned_df_count = normal_sales_cleaned_df.count()


print(f"Total Count: {total_count:,}")
print(f"Total Duplicates: {dup_count}")
print(f"Difference in number of recods: {total_count-dup_count}")
print(f"Distinct Count: {normal_sales_cleaned_df_count:,}")

In [ ]:
# Finding the missing values count for all the columns

count_of_null_values = normal_sales_cleaned_df.select(
    [
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in normal_sales_cleaned_df.columns
    ]
)

display(count_of_null_values)

null_percentage_calculation = count_of_null_values.select(
    [
        col(c)/normal_sales_cleaned_df_count*100.0 for c in count_of_null_values.columns
    ]
)
display(null_percentage_calculation)

In [ ]:
#Checking the Description columns to checks whether the stockcode is present corresponding to it or not in order to decide whether to discard the data or not

display(normal_sales_cleaned_df.filter((col("Description").isNull()) & (col("StockCode").isNotNull())))

#checking where the stockcode is same and description is not null
# Building StockCode-to-Description mapping for investigation

stockcode_description_df = normal_sales_cleaned_df.filter(
    col("Description").isNotNull()
).select(
    "StockCode", "Description"
).distinct()

# Checking StockCodes that have multiple descriptions
stockcode_description_count_df = stockcode_description_df.groupBy(
    "StockCode"
).count().filter(
    col("count") > 1
).orderBy(
    col("count").desc()
)

display(stockcode_description_count_df)

In [ ]:
# Finding StockCodes with exactly one unique description

reliable_stockcode_df = stockcode_description_df.groupBy(
    "StockCode"
).agg(
    countDistinct("Description").alias("description_count")
).filter(
    col("description_count") == 1
).select("StockCode")

# Isolating records with missing descriptions

missing_description_df = normal_sales_cleaned_df.filter(
    col("Description").isNull()
)

# Identifying missing descriptions that can potentially be recovered

recoverable_description_df = missing_description_df.join(
    reliable_stockcode_df,
    on="StockCode",
    how="inner"
)

display(recoverable_description_df)

In [ ]:
reliable_description_lookup_df = stockcode_description_df.groupBy(
    "StockCode"
).agg(
    first("Description").alias("reliable_description")
).join(
    reliable_stockcode_df,
    on="StockCode",
    how="inner"
)

normal_sales_cleaned_df = normal_sales_cleaned_df.join(
    reliable_description_lookup_df,
    on="StockCode",
    how="left"
).withColumn(
    "Description",
    coalesce(col("Description"), col("reliable_description"))
).drop(
    "reliable_description"
)

In [ ]:
#Isolate unrecoverable missing descriptions
remaining_missing_description_df = normal_sales_cleaned_df.filter(
    col("Description").isNull()
)

remaining_missing_description_count = remaining_missing_description_df.count()

print(f"Remaining missing descriptions: {remaining_missing_description_count}")
print(f"Percentage of dataset: {(remaining_missing_description_count / normal_sales_cleaned_df.count()) * 100:.4f}%")

display(remaining_missing_description_df)

In [ ]:
# CHECKING FOR THE MISSING CUSTOMER IDS 
from pyspark.sql.functions import *
employee_count = normal_sales_cleaned_df.filter(
        col("Customer_ID").isNull()
        ).count()

print(f"Total Empty CustomerID : {employee_count}")  

display(
    normal_sales_cleaned_df.filter(
        col("Customer_ID").isNull()
        )
    )



In [ ]:
normal_sales_cleaned_df.createOrReplaceTempView("normal_sales")

mising_customer_df = spark.sql("""
                               select 
                               country,
                               count(*) as missing_customer_records
                               from normal_sales
                               where customer_id is null
                               group by country 
                              
                             order by missing_customer_records desc
                    """)
display(mising_customer_df)

In [ ]:
invoice_customer_analysis_df = spark.sql("""
    SELECT
        Invoice,
        COUNT(*) AS total_records,
        COUNT(DISTINCT Customer_ID) AS distinct_customer_ids,
        COUNT(Customer_ID) AS populated_customer_id_records,
        SUM(CASE WHEN Customer_ID IS NULL THEN 1 ELSE 0 END) AS missing_customer_id_records
    FROM normal_sales
    GROUP BY Invoice
    HAVING SUM(CASE WHEN Customer_ID IS NULL THEN 1 ELSE 0 END) > 0
    ORDER BY missing_customer_id_records DESC
""")

display(invoice_customer_analysis_df)

In [ ]:
invoice_recovery_analysis_df = spark.sql("""
    SELECT
        CASE
            WHEN distinct_customer_ids = 0
                THEN 'No known Customer_ID'
            WHEN distinct_customer_ids = 1
                THEN 'Exactly one known Customer_ID'
            WHEN distinct_customer_ids > 1
                THEN 'Multiple known Customer_IDs'
        END AS recovery_category,
        COUNT(*) AS invoice_count,
        SUM(missing_customer_id_records) AS missing_records
    FROM (
        SELECT
            Invoice,
            COUNT(DISTINCT Customer_ID) AS distinct_customer_ids,
            SUM(CASE WHEN Customer_ID IS NULL THEN 1 ELSE 0 END) AS missing_customer_id_records
        FROM normal_sales
        GROUP BY Invoice
        HAVING SUM(CASE WHEN Customer_ID IS NULL THEN 1 ELSE 0 END) > 0
    ) invoice_analysis
    GROUP BY
        CASE
            WHEN distinct_customer_ids = 0
                THEN 'No known Customer_ID'
            WHEN distinct_customer_ids = 1
                THEN 'Exactly one known Customer_ID'
            WHEN distinct_customer_ids > 1
                THEN 'Multiple known Customer_IDs'
        END
    ORDER BY missing_records DESC
""")

display(invoice_recovery_analysis_df)

In [ ]:
monthly_customer_quality_df = spark.sql("""
    SELECT
        YEAR(InvoiceDate) AS invoice_year,
        MONTH(InvoiceDate) AS invoice_month,
        COUNT(*) AS total_records,
        SUM(CASE WHEN Customer_ID IS NULL THEN 1 ELSE 0 END) AS missing_customer_records,
        ROUND(
            (SUM(CASE WHEN Customer_ID IS NULL THEN 1 ELSE 0 END) * 100.0) / COUNT(*),
            2
        ) AS missing_customer_percentage
    FROM normal_sales
    GROUP BY YEAR(InvoiceDate), MONTH(InvoiceDate)
    ORDER BY invoice_year, invoice_month
""")

display(monthly_customer_quality_df)

In [ ]:
#Checking if the price is negative or 0 or sudden increase in price which may lead to identification of an anomaly
price_quality_df = normal_sales_cleaned_df.filter(
    col("Price") <= 0
)

print("Records with Zero or Negative Price:", price_quality_df.count())

display(price_quality_df)

#The count 0tells that if calculating the revenue it wont be useful if we are looking for quantity * price hence before dropping the price = 0 we will first check if the price is 0 it represents the promotion,sample or gifts
price_quality_df.groupBy(
    "StockCode",
    "Description"
).count().orderBy(
    col("count").desc()
).display()

In [ ]:
price_quality_df.groupBy("Invoice").agg(
    count("*").alias("zero_price_items"),
    sum("Quantity").alias("total_quantity")
).orderBy(
    col("zero_price_items").desc()
).display()

In [ ]:
#Identifying Huge change in sales price to define the outlier
normal_sales_cleaned_df.orderBy(
    col("Quantity").desc()
).select(
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer_ID",
    "Country"
).display()

In [ ]:
#IQR-based outlier detection on Quantity.

quantity_stats = normal_sales_cleaned_df.select(
    percentile_approx("Quantity", 0.25).alias("Q1"),
    percentile_approx("Quantity", 0.75).alias("Q3")
)

quantity_stats.display()

IQR = 12 - 1

lower_bound = 1 - (1.5 * IQR)
upper_bound = 12 + (1.5 * IQR)

quantity_outliers_df = normal_sales_cleaned_df.filter(
    (col("Quantity") < lower_bound) |
    (col("Quantity") > upper_bound)
)

print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("Quantity Outliers:", quantity_outliers_df.count())

quantity_outliers_df.select(
    min("Quantity").alias("Minimum_Outlier_Quantity"),
    max("Quantity").alias("Maximum_Outlier_Quantity"),
    avg("Quantity").alias("Average_Outlier_Quantity"),
    count("*").alias("Total_Outlier_Records")
).display()

In [ ]:
# Text standardization validation:
# This test checks whether textual columns contain leading or trailing whitespaces.
# Such inconsistencies can create separate groups for logically identical values
# and can affect joins, aggregations and downstream analytics.

from pyspark.sql.functions import trim, col

text_columns = ["Invoice", "StockCode", "Description", "Country"]

for column_name in text_columns:
    whitespace_count = normal_sales_cleaned_df.filter(
        col(column_name).isNotNull() &
        (col(column_name) != trim(col(column_name)))
    ).count()

    print(f"{column_name}: {whitespace_count} records with leading/trailing whitespace")

In [ ]:
# Text standardization:
# Remove leading and trailing whitespace from StockCode and Description.
# This prevents logically identical products from being treated as different
# values during grouping, joins, and product-level analytics.
# trim() does not replace NULL values with anything.

from pyspark.sql.functions import trim, col

normal_sales_cleaned_df = normal_sales_cleaned_df.withColumn(
    "StockCode", trim(col("StockCode"))
).withColumn(
    "Description", trim(col("Description"))
)

In [ ]:
# Final Silver validation:
# Validate the cleaned Silver dataset before persistence.
# This checks row count, exact duplicates, and core business rules.
# TotalAmount is excluded because it will be derived in Gold.

total_records = normal_sales_cleaned_df.count()

duplicate_records = total_records - normal_sales_cleaned_df.dropDuplicates().count()

business_violations = normal_sales_cleaned_df.filter(
    (col("Quantity") <= 0) |
    (col("Price") < 0)
).count()

print("Total Silver Records:", total_records)
print("Exact Duplicate Records:", duplicate_records)
print("Business Rule Violations:", business_violations)

In [ ]:
# Persist corrected Silver schema:
# overwrite the existing Silver table so the persisted schema
# no longer contains TotalAmount.

normal_sales_cleaned_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("workspace.ecommerce.silver_retail")

In [ ]:
normal_sales_cleaned_df.printSchema()
checking_retail_data = spark.read.table('workspace.ecommerce.silver_retail')
checking_retail_data.printSchema()